<a href="https://colab.research.google.com/github/jaywong8/Detection-of-Atrial-Fibrillation-Termination/blob/main/notebooks/AgamjotDutta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/jaywong8/Detection-of-Atrial-Fibrillation-Termination.git
%cd Detection-of-Atrial-Fibrillation-Termination

Cloning into 'Detection-of-Atrial-Fibrillation-Termination'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 61 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 21.60 MiB | 10.66 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/Detection-of-Atrial-Fibrillation-Termination


In [2]:
import pandas as pd
from pathlib import Path

DATA = Path("data/10s-30s")
print([p.name for p in DATA.iterdir()])

train = pd.read_csv(DATA / "train_windows_patient_episode_capped_natural.csv")
print(train.shape)
print(train.columns.tolist())
print(train.head())
print(train["label"].value_counts())
print("patients:", train["patient"].nunique())

['test_windows.csv', 'train_windows_patient_episode_capped_natural.csv', 'validation_windows.csv', 'configuration.json', 'pilot_10s_30s-20261029T164953Z-1-001.zip']
(8728, 11)
['patient', 'record_stem', 'episode_id', 'episode_start_sec', 'episode_end_sec', 'prediction_time_sec', 'input_start_sec', 'seconds_until_annotated_end', 'label', 'partition', 'window_id']
  patient record_stem      episode_id  episode_start_sec  episode_end_sec  \
0  p00020  p00020_s13  episode_000378            879.168          912.892   
1  p00020  p00020_s07  episode_000323           2414.248         2460.408   
2  p00020  p00020_s13  episode_000376            170.008          773.008   
3  p00020  p00020_s07  episode_000346           3295.216         3312.780   
4  p00020  p00020_s05  episode_000153           3575.824         3621.936   

   prediction_time_sec  input_start_sec  seconds_until_annotated_end  label  \
0              909.168          899.168                        3.724      1   
1             

In [3]:
!pip install -q wfdb
import wfdb, numpy as np, pandas as pd, time

FS = 250
BEATS = set("NLRBAaJSVrFejnE/fQ?")

def pn_dir(stem):
    pid = stem.split("_")[0]
    return f"icentia11k-continuous-ecg/1.0/{pid[:3]}/{pid}"

def window_features(sig, beat_samples):
    sig = sig[np.isfinite(sig)]
    f = {
        "sig_mean": sig.mean(), "sig_std": sig.std(),
        "sig_rms": np.sqrt(np.mean(sig**2)),
        "sig_min": sig.min(), "sig_max": sig.max(),
        "sig_ptp": np.ptp(sig),
        "line_length": np.mean(np.abs(np.diff(sig))),
        "n_beats": len(beat_samples),
    }
    rr = np.diff(beat_samples) / FS
    if len(rr) >= 3:
        d = np.diff(rr)
        f.update({
            "mean_rr": rr.mean(), "sdnn": rr.std(),
            "rr_cv": rr.std() / rr.mean(),
            "rmssd": np.sqrt(np.mean(d**2)),
            "med_abs_drr": np.median(np.abs(d)),
            "mean_hr": 60 / rr.mean(),
            "pnn50": np.mean(np.abs(d) > 0.05),
        })
    return f

def extract(df):
    rows, fails = [], 0
    for stem, grp in df.groupby("record_stem"):
        try:
            ann = wfdb.rdann(stem, "atr", pn_dir=pn_dir(stem))
            beats = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in BEATS])
        except Exception as e:
            print("ann fail", stem, e); fails += len(grp); continue
        for _, r in grp.iterrows():
            s0 = int(r.input_start_sec * FS); s1 = int(r.prediction_time_sec * FS)
            try:
                rec = wfdb.rdrecord(stem, pn_dir=pn_dir(stem), sampfrom=s0, sampto=s1, channels=[0])
                b = beats[(beats >= s0) & (beats < s1)]
                f = window_features(rec.p_signal[:, 0], b)
                f["window_id"] = r.window_id
                rows.append(f)
            except Exception as e:
                fails += 1
    print("failed windows:", fails)
    return pd.DataFrame(rows)

t = time.time()
test_feats = extract(train.head(30))
print(f"{time.time()-t:.1f}s for 30 windows")
print(test_feats.head())
print(test_feats.isna().sum())

failed windows: 0
10.8s for 30 windows
   sig_mean   sig_std   sig_rms   sig_min   sig_max   sig_ptp  line_length  \
0  0.002313  0.052857  0.052908 -0.188261  0.336180  0.524442     0.009567   
1  0.000850  0.071841  0.071846 -0.188261  0.484100  0.672361     0.011526   
2  0.001565  0.061983  0.062003 -0.201708  0.403417  0.605125     0.011768   
3  0.002141  0.058172  0.058211 -0.121025  0.376522  0.497547     0.009557   
4  0.002652  0.065428  0.065482 -0.174814  0.389969  0.564783     0.010950   

   n_beats   mean_rr      sdnn     rr_cv     rmssd  med_abs_drr     mean_hr  \
0       19  0.531778  0.098229  0.184717  0.142786        0.116  112.829085   
1       18  0.564471  0.115050  0.203819  0.193899        0.172  106.294289   
2       19  0.541333  0.096986  0.179161  0.119178        0.032  110.837438   
3       17  0.570250  0.107842  0.189114  0.177446        0.132  105.217010   
4       18  0.575059  0.078077  0.135773  0.117439        0.058  104.337152   

      pnn50      

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def extract_record(stem, grp):
    rows, fails = [], 0
    try:
        ann = wfdb.rdann(stem, "atr", pn_dir=pn_dir(stem))
        beats = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in BEATS])
    except Exception:
        return rows, len(grp)
    for _, r in grp.iterrows():
        s0 = int(r.input_start_sec * FS); s1 = int(r.prediction_time_sec * FS)
        try:
            rec = wfdb.rdrecord(stem, pn_dir=pn_dir(stem), sampfrom=s0, sampto=s1, channels=[0])
            b = beats[(beats >= s0) & (beats < s1)]
            f = window_features(rec.p_signal[:, 0], b)
            f["window_id"] = r.window_id
            rows.append(f)
        except Exception:
            fails += 1
    return rows, fails

def extract_parallel(df, name, workers=12):
    t = time.time(); rows, fails = [], 0
    groups = list(df.groupby("record_stem"))
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(extract_record, s, g) for s, g in groups]
        for i, fu in enumerate(as_completed(futs), 1):
            r, fl = fu.result(); rows += r; fails += fl
            if i % 25 == 0 or i == len(futs):
                print(f"{name}: {i}/{len(futs)} records, {len(rows)} windows, {time.time()-t:.0f}s")
    feats = pd.DataFrame(rows)
    out = df.merge(feats, on="window_id", how="inner")
    print(f"{name} done: {len(out)} windows, {fails} failed")
    return out

OUT = DATA / "outputs"
OUT.mkdir(exist_ok=True)

val = pd.read_csv(DATA / "validation_windows.csv")
test = pd.read_csv(DATA / "test_windows.csv")

for name, df in [("validation", val), ("test", test), ("train", train)]:
    feats = extract_parallel(df, name)
    feats.to_csv(OUT / f"{name}_features.csv", index=False)
    print("saved", OUT / f"{name}_features.csv")

validation: 25/554 records, 101 windows, 15s
validation: 50/554 records, 239 windows, 27s
validation: 75/554 records, 371 windows, 39s
validation: 100/554 records, 517 windows, 53s
validation: 125/554 records, 650 windows, 63s
validation: 150/554 records, 779 windows, 78s
validation: 175/554 records, 907 windows, 89s
validation: 200/554 records, 1045 windows, 101s
validation: 225/554 records, 1211 windows, 113s
validation: 250/554 records, 1359 windows, 125s
validation: 275/554 records, 1518 windows, 139s
validation: 300/554 records, 1622 windows, 149s
validation: 325/554 records, 1714 windows, 162s
validation: 350/554 records, 1833 windows, 173s
validation: 375/554 records, 1978 windows, 184s
validation: 400/554 records, 2126 windows, 196s
validation: 425/554 records, 2258 windows, 207s
validation: 450/554 records, 2405 windows, 219s
validation: 475/554 records, 2536 windows, 231s
validation: 500/554 records, 2695 windows, 241s
validation: 525/554 records, 2829 windows, 253s
validatio

In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

tr = pd.read_csv(OUT / "train_features.csv")
va = pd.read_csv(OUT / "validation_features.csv")

FEATS = ["sig_mean","sig_std","sig_rms","sig_min","sig_max","sig_ptp","line_length","n_beats","mean_rr","sdnn","rr_cv","rmssd","med_abs_drr","mean_hr","pnn50"]
Xtr, ytr = tr[FEATS], tr["label"]
Xva, yva = va[FEATS], va["label"]

print(f"positive rate  train: {ytr.mean():.2f}  validation: {yva.mean():.2f}")

models = {
    "Logistic Regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=42)),
}

results = []
for name, m in models.items():
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xva)[:, 1]
    tn, fp, fn, tp = confusion_matrix(yva, (p >= 0.5).astype(int)).ravel()
    results.append({
        "model": name,
        "ROC-AUC": roc_auc_score(yva, p),
        "PR-AUC": average_precision_score(yva, p),
        "sensitivity": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "precision": tp / (tp + fp),
    })

print(pd.DataFrame(results).round(3).to_string(index=False))

rf = models["Random Forest"][-1]
print("\nRF top features:")
print(pd.Series(rf.feature_importances_, index=FEATS).sort_values(ascending=False).round(3).head(8))

positive rate  train: 0.55  validation: 0.18
              model  ROC-AUC  PR-AUC  sensitivity  specificity  precision
Logistic Regression    0.604   0.318        0.687        0.434      0.210
      Random Forest    0.588   0.259        0.630        0.451      0.201

RF top features:
sig_mean       0.102
mean_hr        0.088
mean_rr        0.088
line_length    0.083
sig_min        0.071
sig_rms        0.071
sig_std        0.069
sig_max        0.066
dtype: float64


In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier

for d in (tr, va):
    d["time_in_episode"] = d["prediction_time_sec"] - d["episode_start_sec"]

RR = ["n_beats","mean_rr","sdnn","rr_cv","rmssd","med_abs_drr","mean_hr","pnn50"]
CLEAN_SIG = ["sig_std","line_length"]
feature_sets = {
    "RR only": RR,
    "RR + time": RR + ["time_in_episode"],
    "RR + time + clean signal": RR + ["time_in_episode"] + CLEAN_SIG,
}

def make_models():
    return {
        "LogReg": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
        "RandomForest": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced", n_jobs=-1, random_state=42)),
        "GradBoost": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, class_weight="balanced", random_state=42),
    }

rows = []
for fs_name, cols in feature_sets.items():
    for m_name, m in make_models().items():
        m.fit(tr[cols], tr["label"])
        p = m.predict_proba(va[cols])[:, 1]
        rows.append({"features": fs_name, "model": m_name, "ROC-AUC": roc_auc_score(va["label"], p), "PR-AUC": average_precision_score(va["label"], p)})

print(pd.DataFrame(rows).sort_values("PR-AUC", ascending=False).round(3).to_string(index=False))

                features        model  ROC-AUC  PR-AUC
               RR + time       LogReg    0.778   0.441
RR + time + clean signal       LogReg    0.773   0.433
               RR + time RandomForest    0.737   0.350
RR + time + clean signal RandomForest    0.732   0.339
                 RR only       LogReg    0.623   0.336
               RR + time    GradBoost    0.734   0.334
RR + time + clean signal    GradBoost    0.731   0.325
                 RR only RandomForest    0.594   0.300
                 RR only    GradBoost    0.607   0.294


In [7]:
m = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")).fit(tr[["time_in_episode"]], tr["label"]); p = m.predict_proba(va[["time_in_episode"]])[:, 1]; print("time only  ROC-AUC:", round(roc_auc_score(va["label"], p), 3), " PR-AUC:", round(average_precision_score(va["label"], p), 3))

time only  ROC-AUC: 0.776  PR-AUC: 0.419


Got the baselines running on the 10s/30s windows. Signal + RR features alone are pretty weak (~0.6 ROC-AUC), but adding time in episode gets logistic regression to 0.78 ROC-AUC / 0.44 PR-AUC. Time alone gets almost the same though, so RR isn’t adding much.